# Model A vs B vs C — Head-to-Head Comparison

A **single, controlled experiment** that trains all three models under *identical*
settings on the same data split, then scores them with the **same metric suite**
so the comparison is fair.

| Model | Structure signal | How it is used |
|-------|------------------|----------------|
| **A** | none | pixel L1 to class/real image |
| **B** | MediaPipe 21 landmarks | landmark MSE *loss* (frozen regressor) |
| **C** | edge + silhouette + distance map | per-image *conditioning input* (pix2pix) |

**Fair metrics computed for all three:**
- **FID** ↓ — distributional realism
- **Recognition accuracy** ↑ — a classifier trained on *real* images, tested on each model's *generated* images (the most meaningful sign-language metric)
- **Diversity** ↑ — intra-class spread

> Defaults run a **fast but real** comparison (subset of classes, few epochs).
> Scale `NUM_SUBSET_CLASSES` and `EPOCHS` up for the full benchmark.
> ⚠️ Requires a **GPU runtime** (Runtime → Change runtime type → GPU).

## 1 — Install + download dataset

In [ ]:
!pip install -q tensorflow==2.19.0 mediapipe==0.10.14 opencv-python pytorch-fid scikit-image scikit-learn lpips kagglehub
print("installed")

In [ ]:
# ── Download ArASL ──────────────────────────────────────────────────────────
# Easiest on Colab: kagglehub (prompts for a Kaggle token the first time, or set
# KAGGLE creds). If the slug differs / you already have the data on Drive, just
# set DATA_PATH manually to the folder that contains the 32 class subfolders.
import os, glob
DATA_PATH = None
try:
    import kagglehub
    # ArASL 2018 (54K). If this slug 404s, replace it with the dataset you use.
    p = kagglehub.dataset_download("mhantor/arabic-sign-language")
    # find the folder that actually contains class subfolders
    cands = [d for d, subs, files in os.walk(p) if sum(os.path.isdir(os.path.join(d,s)) for s in subs) >= 10]
    DATA_PATH = sorted(cands, key=lambda d: -sum(1 for _ in glob.glob(d+"/*/")))[0]
    print("Dataset at:", DATA_PATH)
except Exception as e:
    print("kagglehub failed:", e)
    print("=> Set DATA_PATH manually (e.g. mount Drive) to the folder with class subfolders.")
    # from google.colab import drive; drive.mount('/content/drive')
    # DATA_PATH = '/content/drive/MyDrive/ArASL_Database_54K_Final/ArASL_Database_54K_Final'
assert DATA_PATH and os.path.isdir(DATA_PATH), "Set DATA_PATH to the dataset folder."
print("classes found:", len([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH,d))]))

## 2 — Config (same for all three models)

In [ ]:
import numpy as np, tensorflow as tf, cv2, json, warnings
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

# ── experiment scale (raise for the full benchmark) ─────────────────────────
NUM_SUBSET_CLASSES = 6      # use the N largest classes; set None for ALL 32
MAX_PER_CLASS      = 800    # cap images/class to keep the demo fast; None = all
EPOCHS             = 15
# ── shared hyperparameters ──────────────────────────────────────────────────
Z_DIM, IMG_SIZE, BATCH_SIZE = 128, 128, 32
LR_G, LR_D, LABEL_SMOOTH = 2e-4, 1e-4, 0.9
USE_MIXED_PRECISION, USE_XLA = True, False
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)
_gpus = tf.config.list_physical_devices("GPU")
for g in _gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except: pass
if USE_MIXED_PRECISION and _gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16"); print("mixed_float16 ON")
print("GPU:", _gpus)

## 3 — Load data (shared split for all models)

In [ ]:
from tqdm import tqdm
from collections import Counter
all_classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH,d))])
# pick the N largest classes for the subset
counts = {c: len(os.listdir(os.path.join(DATA_PATH,c))) for c in all_classes}
classes = sorted(counts, key=lambda c:-counts[c])[:NUM_SUBSET_CLASSES] if NUM_SUBSET_CLASSES else all_classes
print("Using classes:", classes)

imgs, labs = [], []
for c in tqdm(classes, desc="load"):
    fs = sorted(os.listdir(os.path.join(DATA_PATH,c)))
    if MAX_PER_CLASS: fs = fs[:MAX_PER_CLASS]
    for fn in fs:
        if not fn.lower().endswith((".png",".jpg",".jpeg")): continue
        im = cv2.imread(os.path.join(DATA_PATH,c,fn), cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        im = cv2.resize(im, (IMG_SIZE, IMG_SIZE))
        imgs.append((im.astype(np.float32)-127.5)/127.5); labs.append(c)
images = np.asarray(imgs, np.float32)[..., None]
enc = LabelEncoder(); labels = enc.fit_transform(labs).astype(np.int64)
num_classes = len(enc.classes_)
rng = np.random.default_rng(RANDOM_SEED); perm = rng.permutation(len(images))
images, labels = images[perm], labels[perm]
# train / eval split
n_eval = max(num_classes*30, int(0.1*len(images)))
X_tr, y_tr = images[:-n_eval], labels[:-n_eval]
X_ev, y_ev = images[-n_eval:], labels[-n_eval:]
print("train", X_tr.shape, "eval", X_ev.shape, "classes", num_classes)

## 4 — Structure maps (Model C) + MediaPipe landmarks (Model B)

In [ ]:
# Model C condition: edge + silhouette + distance-transform (100% coverage)
def structure_map(img_norm):
    g = ((img_norm[:,:,0]+1)*127.5).clip(0,255).astype(np.uint8)
    edge = cv2.Canny(g, 60, 160)
    _, sil = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    if sil.mean() > 127: sil = 255 - sil
    dist = cv2.normalize(cv2.distanceTransform(sil, cv2.DIST_L2, 3), None, 0, 255, cv2.NORM_MINMAX)
    return (np.stack([edge, sil, dist], -1).astype(np.float32)/127.5)-1.0

C_tr = np.stack([structure_map(x) for x in tqdm(X_tr, desc="struct")]).astype(np.float32)
print("condition maps:", C_tr.shape, "coverage", np.mean([np.any(c>-0.99) for c in C_tr]))

In [ ]:
# Model B landmarks via a REUSED MediaPipe detector (optimized)
import mediapipe as mp
class Ext:
    def __init__(s):
        H=mp.solutions.hands.Hands
        s.hi=H(static_image_mode=True,max_num_hands=1,min_detection_confidence=0.3,model_complexity=1)
        s.lo=H(static_image_mode=True,max_num_hands=1,min_detection_confidence=0.15,model_complexity=1)
    def _c(s,u8):
        e=cv2.createCLAHE(3.0,(4,4)).apply(u8)
        if e.mean()<100: e=cv2.bitwise_not(e)
        f=e.astype(np.float32)/255
        return np.stack([np.clip(f*210+40,0,255),np.clip(f*170+25,0,255),np.clip(f*140+10,0,255)],-1).astype(np.uint8)
    def _pp(s,u8,t):
        p=int(max(u8.shape)*0.15); u8=cv2.copyMakeBorder(u8,p,p,p,p,cv2.BORDER_CONSTANT,value=int(u8.mean()))
        return s._c(cv2.resize(u8,(t,t),interpolation=cv2.INTER_LANCZOS4))
    def _r(s,res):
        if res.multi_hand_landmarks:
            return np.array([[p.x,p.y,p.z] for p in res.multi_hand_landmarks[0].landmark],np.float32).flatten()
        return None
    def extract(s,img):
        u8=((img[:,:,0]+1)*127.5).clip(0,255).astype(np.uint8)
        for det,t in [(s.hi,256),(s.hi,320),(s.lo,256)]:
            r=s._r(det.process(s._pp(u8,t)))
            if r is not None: return r
        return np.zeros(63,np.float32)
    def close(s): s.hi.close(); s.lo.close()

ext=Ext(); LM_tr=np.stack([ext.extract(x) for x in tqdm(X_tr,desc="mediapipe")]).astype(np.float32); ext.close()
print("landmark detection rate:", LM_tr.any(axis=1).mean())

## 5 — Shared backbone + optimizer helpers

In [ ]:
def make_opt(lr):
    o=tf.keras.optimizers.Adam(lr,beta_1=0.5,clipnorm=1.0)
    return tf.keras.mixed_precision.LossScaleOptimizer(o) if USE_MIXED_PRECISION else o
def apply_loss(opt,loss,tape,vs):
    if USE_MIXED_PRECISION:
        g=tape.gradient(opt.scale_loss(loss),vs)
        opt.apply(g,vs)   # Keras 3 unscales internally
    else:
        g=tape.gradient(loss,vs)
        opt.apply_gradients(zip(g,vs))
bce=tf.keras.losses.BinaryCrossentropy(from_logits=True)

def gen_block(x,f):
    x=layers.Conv2DTranspose(f,4,2,padding="same",use_bias=False)(x)
    return layers.ReLU()(layers.BatchNormalization()(x))

def build_G(num_classes):                         # A & B: noise+label -> image
    n=tf.keras.Input((Z_DIM,)); l=tf.keras.Input((num_classes,))
    le=layers.LeakyReLU(0.2)(layers.Dense(128,use_bias=False)(l))
    x=layers.Dense(4*4*512,use_bias=False)(layers.Concatenate()([n,le]))
    x=layers.ReLU()(layers.BatchNormalization()(x)); x=layers.Reshape((4,4,512))(x)
    x=gen_block(x,256); x=gen_block(x,128); x=gen_block(x,128); x=gen_block(x,64); x=gen_block(x,32)
    o=layers.Activation("tanh",dtype="float32")(layers.Conv2D(1,3,padding="same",dtype="float32")(x))
    return tf.keras.Model([n,l],o)

def build_D(num_classes, in_ch=1):                # A & B: (image,label) -> logit
    SN=layers.SpectralNormalization
    img=tf.keras.Input((IMG_SIZE,IMG_SIZE,in_ch)); l=tf.keras.Input((num_classes,))
    lp=layers.Reshape((IMG_SIZE,IMG_SIZE,1))(layers.Dense(IMG_SIZE*IMG_SIZE)(l))
    x=layers.Concatenate()([img,lp])
    for f in [64,128,256,512,512]:
        x=layers.LeakyReLU(0.2)(SN(layers.Conv2D(f,4,2,padding="same"))(x))
    o=layers.Dense(1,dtype="float32")(layers.Flatten()(x))
    return tf.keras.Model([img,l],o)

In [ ]:
# Model C conditioned generator (encoder-decoder) + paired discriminator
def build_G_cond(num_classes):                    # (condition,label,noise) -> image
    c=tf.keras.Input((IMG_SIZE,IMG_SIZE,3)); l=tf.keras.Input((num_classes,)); n=tf.keras.Input((Z_DIM,))
    e=c
    for f in [64,128,256,512]:
        e=layers.LeakyReLU(0.2)(layers.BatchNormalization()(layers.Conv2D(f,4,2,padding="same",use_bias=False)(e)))
    e=layers.Flatten()(e)                                   # 8*8*512
    le=layers.LeakyReLU(0.2)(layers.Dense(128,use_bias=False)(l))
    x=layers.Dense(8*8*512,use_bias=False)(layers.Concatenate()([e,n,le]))
    x=layers.ReLU()(layers.BatchNormalization()(x)); x=layers.Reshape((8,8,512))(x)
    x=gen_block(x,256); x=gen_block(x,128); x=gen_block(x,64); x=gen_block(x,32)   # 8->128
    o=layers.Activation("tanh",dtype="float32")(layers.Conv2D(1,3,padding="same",dtype="float32")(x))
    return tf.keras.Model([c,l,n],o)

def build_D_cond(num_classes):                    # (image,condition,label) -> logit
    SN=layers.SpectralNormalization
    img=tf.keras.Input((IMG_SIZE,IMG_SIZE,1)); c=tf.keras.Input((IMG_SIZE,IMG_SIZE,3)); l=tf.keras.Input((num_classes,))
    lp=layers.Reshape((IMG_SIZE,IMG_SIZE,1))(layers.Dense(IMG_SIZE*IMG_SIZE)(l))
    x=layers.Concatenate()([img,c,lp])
    for f in [64,128,256,512,512]:
        x=layers.LeakyReLU(0.2)(SN(layers.Conv2D(f,4,2,padding="same"))(x))
    o=layers.Dense(1,dtype="float32")(layers.Flatten()(x))
    return tf.keras.Model([img,c,l],o)

def build_regressor():                            # Model B frozen landmark proxy
    i=tf.keras.Input((IMG_SIZE,IMG_SIZE,1)); x=i
    for f in [32,64,128,256]:
        x=layers.LeakyReLU(0.2)(layers.Conv2D(f,3,2,padding="same")(x))
    x=layers.Dropout(0.3)(layers.LeakyReLU(0.2)(layers.Dense(256)(layers.GlobalAveragePooling2D()(x))))
    return tf.keras.Model(i,layers.Dense(63,activation="sigmoid",dtype="float32")(x))

## 6 — Train the three models (identical schedule)

In [ ]:
def onehot(y): return tf.one_hot(y, num_classes)

def make_ds(arrays):
    ds=tf.data.Dataset.from_tensor_slices(arrays).shuffle(len(arrays[0]),seed=RANDOM_SEED,reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE,drop_remainder=True).prefetch(tf.data.AUTOTUNE)

def train_AB(use_landmarks):
    G,D=build_G(num_classes),build_D(num_classes); go,do=make_opt(LR_G),make_opt(LR_D)
    reg=None
    if use_landmarks:
        reg=build_regressor(); v=LM_tr.any(1)
        reg.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="mse")
        reg.fit(X_tr[v],np.clip(LM_tr[v],0,1),epochs=8,batch_size=64,verbose=0); reg.trainable=False
    ds=make_ds((X_tr,y_tr,LM_tr)) if use_landmarks else make_ds((X_tr,y_tr,np.zeros((len(X_tr),1),np.float32)))
    @tf.function(jit_compile=USE_XLA)
    def step(real,y,lm):
        oh=onehot(y); nz=tf.random.normal([tf.shape(real)[0],Z_DIM])
        with tf.GradientTape() as t:
            fake=G([nz,oh],training=True)
            d_loss=bce(tf.ones_like(D([real,oh]))*LABEL_SMOOTH,D([real,oh],training=True))+bce(tf.zeros_like(D([fake,oh])),D([fake,oh],training=True))
        apply_loss(do,d_loss,t,D.trainable_variables)
        nz=tf.random.normal([tf.shape(real)[0],Z_DIM])
        with tf.GradientTape() as t:
            fake=G([nz,oh],training=True); f=D([fake,oh],training=True)
            g=bce(tf.ones_like(f),f)+5.0*tf.reduce_mean(tf.abs(fake-real))
            if use_landmarks:
                pl=reg(fake,training=False); val=tf.cast(tf.reduce_any(lm!=0.,1,keepdims=True),tf.float32)
                g+=2.0*tf.reduce_mean(tf.square(pl-lm)*val)
        apply_loss(go,g,t,G.trainable_variables)
        return d_loss,g
    for ep in range(EPOCHS):
        dl=gl=k=0
        for real,y,lm in ds:
            d,g=step(real,y,lm); dl+=float(d); gl+=float(g); k+=1
        print(f"  ep{ep+1}/{EPOCHS} D={dl/k:.3f} G={gl/k:.3f}")
    return G

def train_C():
    G,D=build_G_cond(num_classes),build_D_cond(num_classes); go,do=make_opt(LR_G),make_opt(LR_D)
    ds=make_ds((X_tr,y_tr,C_tr))
    @tf.function(jit_compile=USE_XLA)
    def step(real,y,cond):
        oh=onehot(y); nz=tf.random.normal([tf.shape(real)[0],Z_DIM])
        with tf.GradientTape() as t:
            fake=G([cond,oh,nz],training=True)
            d_loss=bce(tf.ones_like(D([real,cond,oh]))*LABEL_SMOOTH,D([real,cond,oh],training=True))+bce(tf.zeros_like(D([fake,cond,oh])),D([fake,cond,oh],training=True))
        apply_loss(do,d_loss,t,D.trainable_variables)
        nz=tf.random.normal([tf.shape(real)[0],Z_DIM])
        with tf.GradientTape() as t:
            fake=G([cond,oh,nz],training=True); f=D([fake,cond,oh],training=True)
            g=bce(tf.ones_like(f),f)+5.0*tf.reduce_mean(tf.abs(fake-real))   # PAIRED L1 (aligned)
        apply_loss(go,g,t,G.trainable_variables)
        return d_loss,g
    for ep in range(EPOCHS):
        dl=gl=k=0
        for real,y,cond in ds:
            d,g=step(real,y,cond); dl+=float(d); gl+=float(g); k+=1
        print(f"  ep{ep+1}/{EPOCHS} D={dl/k:.3f} G={gl/k:.3f}")
    return G

print("=== Training Model A ==="); G_A=train_AB(False)
print("=== Training Model B ==="); G_B=train_AB(True)
print("=== Training Model C ==="); G_C=train_C()

## 7 — Unified evaluation (same metrics for all three)

In [ ]:
# A small classifier trained on REAL images -> "recognition accuracy" of fakes
clf=tf.keras.Sequential([tf.keras.Input((IMG_SIZE,IMG_SIZE,1)),
    layers.Conv2D(32,3,2,"same"),layers.LeakyReLU(0.2),
    layers.Conv2D(64,3,2,"same"),layers.LeakyReLU(0.2),
    layers.Conv2D(128,3,2,"same"),layers.LeakyReLU(0.2),
    layers.GlobalAveragePooling2D(),layers.Dense(num_classes,dtype="float32")])
clf.compile(optimizer="adam",loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=["accuracy"])
clf.fit(X_tr,y_tr,epochs=8,batch_size=64,validation_data=(X_ev,y_ev),verbose=0)
print("classifier real eval-acc:", float(clf.evaluate(X_ev,y_ev,verbose=0)[1]))

def gen_for_eval(kind,G,n_per=40,seed=0):
    tf.random.set_seed(seed); out,ys=[],[]
    for c in range(num_classes):
        oh=onehot(np.full(n_per,c)); nz=tf.random.normal([n_per,Z_DIM],seed=seed*100+c)
        if kind=="C":
            pool=np.where(y_tr==c)[0]; idx=np.random.default_rng(seed+c).choice(pool,n_per)
            f=G([C_tr[idx],oh,nz],training=False).numpy()
        else:
            f=G([nz,oh],training=False).numpy()
        out.append(f); ys.append(np.full(n_per,c))
    return np.concatenate(out), np.concatenate(ys)

def recognition(kind,G):
    f,ys=gen_for_eval(kind,G)
    pred=clf.predict(f,verbose=0).argmax(1)
    return float((pred==ys).mean())

def diversity(kind,G,n_per=12,seed=1):
    tf.random.set_seed(seed); ds=[]
    for c in range(num_classes):
        oh=onehot(np.full(n_per,c)); nz=tf.random.normal([n_per,Z_DIM],seed=seed+c)
        if kind=="C":
            pool=np.where(y_tr==c)[0]; idx=np.random.default_rng(seed+c).choice(pool,n_per)
            f=G([C_tr[idx],oh,nz],training=False).numpy().reshape(n_per,-1)
        else:
            f=G([nz,oh],training=False).numpy().reshape(n_per,-1)
        ds+=[np.mean(np.abs(f[i]-f[j])) for i in range(n_per) for j in range(i+1,n_per)]
    return float(np.mean(ds))

In [ ]:
# FID (pytorch-fid) on saved real vs generated images
from pytorch_fid import fid_score; import torch
from PIL import Image
DEV="cuda" if torch.cuda.is_available() else "cpu"
def save_imgs(arr,d):
    os.makedirs(d,exist_ok=True)
    for i,im in enumerate(arr):
        u=(im[:,:,0]*127.5+127.5).clip(0,255).astype(np.uint8)
        Image.fromarray(np.stack([u]*3,-1)).save(f"{d}/{i:05d}.png")
save_imgs(X_ev,"/content/fid_real")
def fid(kind,G):
    f,_=gen_for_eval(kind,G,n_per=max(40,len(X_ev)//num_classes))
    save_imgs(f,f"/content/fid_{kind}")
    return fid_score.calculate_fid_given_paths(["/content/fid_real",f"/content/fid_{kind}"],32,DEV,2048)

results={}
for kind,G in [("A",G_A),("B",G_B),("C",G_C)]:
    results[kind]={"FID":fid(kind,G),"Recognition":recognition(kind,G),"Diversity":diversity(kind,G)}
    print(kind,results[kind])

## 8 — Comparison table + chart

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df=pd.DataFrame(results).T[["FID","Recognition","Diversity"]]
df.index=["A (no MP)","B (MediaPipe)","C (structure-cond)"]
print(df.round(4).to_string())
df.to_json("/content/comparison_ABC.json")

fig,ax=plt.subplots(1,3,figsize=(15,4))
for i,(m,better) in enumerate([("FID","lower"),("Recognition","higher"),("Diversity","higher")]):
    df[m].plot.bar(ax=ax[i],color=["#5ba4e8","#e8a450","#3dba8c"]); ax[i].set_title(f"{m} ({better} better)"); ax[i].grid(alpha=.3,axis="y")
plt.tight_layout(); plt.savefig("/content/comparison_ABC.png",dpi=150,bbox_inches="tight"); plt.show()

best_fid=df["FID"].idxmin(); best_rec=df["Recognition"].idxmax()
print(f"\\nLowest FID: {best_fid} | Best recognition: {best_rec}")
print("Note: small-subset/few-epoch run -> treat as a directional signal; scale up to confirm.")